# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall   
from catboost import CatBoostClassifier

In [ ]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

In [ ]:
events.head()

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [ ]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [ ]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

## Простейшие признаки

Два агрегата — сколько событий и сколько разных объявлений. Этого заведомо мало.

In [ ]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [ ]:
is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['n_events', 'item_nunique']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

## Сабмит

In [ ]:
model.fit(Xtr[cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.

In [ ]:
# количество событий каждого типа для каждого cookie
event_counts = (
    ev_tr
    .pivot_table(
        index='cookie_id',
        columns='event_name',
        values='eid',
        aggfunc='count',
        fill_value=0
    )
    .add_suffix('_n')
    .reset_index()
)

event_counts.head()

In [ ]:
# количество событий каждого типа для каждого cookie в тестовой выборке.
event_counts_te = (
    ev_te
    .pivot_table(
        index='cookie_id',
        columns='event_name',
        values='eid',
        aggfunc='count',
        fill_value=0
    )
    .add_suffix('_n')
    .reset_index()
)

event_counts_te.head()

In [ ]:
# объединение признаков количества событий с основной выборкой
Xtr2 = Xtr.merge(event_counts, on='cookie_id', how='left').fillna(0)
Xte2 = Xte.merge(event_counts_te, on='cookie_id', how='left').fillna(0)

print(Xtr2.shape)
print(Xte2.shape)

Xtr2.head()

In [ ]:
# обучение Random Forest и проверка качества признаков типов событий на валидации
cols2 = [c for c in Xtr2.columns if c != 'cookie_id']

model2 = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0
)

model2.fit(Xtr2.loc[~is_valid, cols2], ytr[~is_valid])

p_va2 = model2.predict_proba(
    Xtr2.loc[is_valid, cols2]
)[:, 1]

print(
    'P@R0.7 с признаками типов событий:',
    round(precision_at_recall(ytr[is_valid], p_va2), 4)
)

In [ ]:
# рассчет временных характеристик активности каждого cookie
def time_features(ev):
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()

    g = ev.groupby('cookie_id')['event_ts']

    f = pd.DataFrame({
        'active_seconds': (g.max() - g.min()).dt.total_seconds(),
        'mean_gap_seconds': g.apply(
            lambda x: x.diff().dt.total_seconds().mean()
        ),
        'min_gap_seconds': g.apply(
            lambda x: x.diff().dt.total_seconds().min()
        ),
        'max_gap_seconds': g.apply(
            lambda x: x.diff().dt.total_seconds().max()
        )
    })

    return f.reset_index()

time_tr = time_features(ev_tr)
time_te = time_features(ev_te)

time_tr.head()

In [ ]:
# добавка временных признаков и проверка их влияния на качество модели
Xtr3 = Xtr2.merge(time_tr, on='cookie_id', how='left').fillna(0)
Xte3 = Xte2.merge(time_te, on='cookie_id', how='left').fillna(0)

cols3 = [c for c in Xtr3.columns if c != 'cookie_id']

model3 = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0
)

model3.fit(
    Xtr3.loc[~is_valid, cols3],
    ytr[~is_valid]
)

p_va3 = model3.predict_proba(
    Xtr3.loc[is_valid, cols3]
)[:, 1]

print(
    'P@R0.7 с временными признаками:',
    round(precision_at_recall(ytr[is_valid], p_va3), 4)
)

In [ ]:
# рассчет разнообразия действий: объект, категория, локация, поисковых запросов и страниц
def diversity_features(ev):
    g = ev.groupby('cookie_id')

    f = pd.DataFrame({
        'item_nunique': g['item_id'].nunique(),
        'category_nunique': g['item_category'].nunique(),
        'location_nunique': g['item_location'].nunique(),
        'search_query_nunique': g['search_query'].nunique(),
        'search_page_nunique': g['search_page'].nunique(),
    })

    return f.reset_index()

div_tr = diversity_features(ev_tr)
div_te = diversity_features(ev_te)

div_tr.head()

In [ ]:
# плюс признаки разнообразного поведения
Xtr4 = Xtr3.merge(
    div_tr.drop(columns='item_nunique'),
    on='cookie_id',
    how='left'
).fillna(0)

Xte4 = Xte3.merge(
    div_te.drop(columns='item_nunique'),
    on='cookie_id',
    how='left'
).fillna(0)

cols4 = [c for c in Xtr4.columns if c != 'cookie_id']

print(Xtr4.shape)
print(Xte4.shape)

In [ ]:
# обучение модели с признаками разнообр поведения
model4 = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0
)

model4.fit(
    Xtr4.loc[~is_valid, cols4],
    ytr[~is_valid]
)

p_va4 = model4.predict_proba(
    Xtr4.loc[is_valid, cols4]
)[:, 1]

print(
    'P@R0.7 с признаками разнообразия:',
    round(precision_at_recall(ytr[is_valid], p_va4), 4)
)

In [ ]:
# проверка на дубликаты событий
print('Всего событий:', len(events))
print('Полных дубликатов:', events.duplicated().sum())
print('Дубликатов по eid:', events['eid'].duplicated().sum())

In [ ]:
# проверка полных дубликатов
duplicates = events[events.duplicated(keep=False)].sort_values(
    ['cookie_id', 'event_ts']
)

duplicates.head(10)

In [ ]:
# удаление только полных дубликатов
events_clean = events.drop_duplicates().copy()

print('Было событий:', len(events))
print('Стало событий:', len(events_clean))
print('Удалено:', len(events) - len(events_clean))

In [ ]:
# только события внутри окон наблюдения
ev_tr_clean = events_in_window(events_clean, train)
ev_te_clean = events_in_window(events_clean, test)

# пересчет всех групп признаков
Xtr_clean = basic_features(ev_tr_clean, train)
Xte_clean = basic_features(ev_te_clean, test)

event_counts_clean_tr = (
    ev_tr_clean
    .pivot_table(
        index='cookie_id',
        columns='event_name',
        values='eid',
        aggfunc='count',
        fill_value=0
    )
    .add_suffix('_n')
    .reset_index()
)

event_counts_clean_te = (
    ev_te_clean
    .pivot_table(
        index='cookie_id',
        columns='event_name',
        values='eid',
        aggfunc='count',
        fill_value=0
    )
    .add_suffix('_n')
    .reset_index()
)

time_clean_tr = time_features(ev_tr_clean)
time_clean_te = time_features(ev_te_clean)

div_clean_tr = diversity_features(ev_tr_clean)
div_clean_te = diversity_features(ev_te_clean)

print(len(ev_tr_clean), len(ev_te_clean))

In [ ]:
# сборка признаков на очищенных данных
Xtr_clean = (
    Xtr_clean
    .merge(event_counts_clean_tr, on='cookie_id', how='left')
    .merge(time_clean_tr, on='cookie_id', how='left')
    .merge(
        div_clean_tr.drop(columns='item_nunique'),
        on='cookie_id',
        how='left'
    )
    .fillna(0)
)

Xte_clean = (
    Xte_clean
    .merge(event_counts_clean_te, on='cookie_id', how='left')
    .merge(time_clean_te, on='cookie_id', how='left')
    .merge(
        div_clean_te.drop(columns='item_nunique'),
        on='cookie_id',
        how='left'
    )
    .fillna(0)
)

cols_clean = [c for c in Xtr_clean.columns if c != 'cookie_id']

print(Xtr_clean.shape)
print(Xte_clean.shape)

In [ ]:
# обучение на признаках из очищенных событий
model_clean = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0
)

model_clean.fit(
    Xtr_clean.loc[~is_valid, cols_clean],
    ytr[~is_valid]
)

p_va_clean = model_clean.predict_proba(
    Xtr_clean.loc[is_valid, cols_clean]
)[:, 1]

print(
    'P@R0.7 после удаления дубликатов:',
    round(
        precision_at_recall(ytr[is_valid], p_va_clean),
        4
    )
)

In [ ]:
# медианный интервал между последовательными событиями для каждого куки
# показывает типичный темп взаимодействия пользователя с сервисом
def median_gap_features(ev):
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()

    gaps = (
        ev.groupby('cookie_id')['event_ts']
        .diff()
        .dt.total_seconds()
    )

    f = (
        gaps.groupby(ev['cookie_id'])
        .median()
        .reset_index(name='median_gap_seconds')
    )

    return f

median_gap_tr = median_gap_features(ev_tr_clean)
median_gap_te = median_gap_features(ev_te_clean)

Xtr_median = Xtr_clean.merge(
    median_gap_tr,
    on='cookie_id',
    how='left'
).fillna(0)

Xte_median = Xte_clean.merge(
    median_gap_te,
    on='cookie_id',
    how='left'
).fillna(0)

cols_median = [
    c for c in Xtr_median.columns
    if c != 'cookie_id'
]

model_median = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0
)

model_median.fit(
    Xtr_median.loc[~is_valid, cols_median],
    ytr[~is_valid]
)

p_va_median = model_median.predict_proba(
    Xtr_median.loc[is_valid, cols_median]
)[:, 1]

print(
    'P@R0.7:',
    round(precision_at_recall(ytr[is_valid], p_va_median), 4)
)

In [ ]:
# финальная модель на всех данных и вероятности на тест куки
cat_final = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=0,
    verbose=False,
    thread_count=-1
)

cat_final.fit(
    Xtr_median[cols_median],
    ytr
)

test_scores_cat = cat_final.predict_proba(
    Xte_median[cols_median]
)[:, 1]

submission_cat = pd.DataFrame({
    'cookie_id': Xte_median['cookie_id'],
    'score': test_scores_cat
})

assert len(submission_cat) == len(test)
assert submission_cat['cookie_id'].nunique() == len(test)
assert submission_cat['score'].notna().all()
assert submission_cat['score'].between(0, 1).all()

submission_cat.to_csv('submission.csv', index=False)

print('Размер:', submission_cat.shape)
print('Дубликаты:', submission_cat['cookie_id'].duplicated().sum())
print('Пропуски:', submission_cat.isna().sum().sum())
print('Score вне [0, 1]:', (~submission_cat['score'].between(0, 1)).sum())
print('Средний score:', round(submission_cat['score'].mean(), 4))

In [ ]:
check_submission = pd.read_csv('submission.csv')

print('Размер:', check_submission.shape)
print('Колонки:', check_submission.columns.tolist())
print('Дубликаты cookie_id:', check_submission['cookie_id'].duplicated().sum())
print('Пропуски:', check_submission.isna().sum().sum())
print('Score вне диапазона 0–1:',
      ((check_submission['score'] < 0) |
       (check_submission['score'] > 1)).sum())

print('\nПервые строки:')
print(check_submission.head())

In [ ]:
requirements = """numpy==2.5.3
pandas==3.0.6
scikit-learn==1.9.1
scipy==1.18.1
catboost==1.2.10
"""

with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements)

print('requirements.txt сохранён')

In [ ]:
readme = """# Avito Data Science Bootcamp — тестовое задание

## Задача

Определить вероятности того, что cookie относится к автоматизированному
трафику, на основе событий внутри заданного окна наблюдения.

## Подход

1. Загрузил train, test и события.
2. Для каждого cookie оставил только события, попадающие в его окно наблюдения.
3. Проверил данные на дубликаты. Нашёл 4863 полных дубликата событий и удалил их.
4. Построил признаки:
   - общее количество событий;
   - количество событий каждого типа;
   - количество уникальных объектов, категорий, локаций и поисковых запросов;
   - продолжительность активности;
   - средний, минимальный и максимальный интервал между событиями;
   - медианный интервал между событиями.
5. В качестве модели использовал CatBoostClassifier.
6. Для валидации использовал временное разделение: записи с началом окна
   до 17 апреля 2026 года использовались для обучения, более поздние —
   для валидации.
7. Основная метрика — Precision при Recall не менее 70%.

## Эксперименты

Базовая модель с двумя признаками дала P@R0.7 = 0.1025.
Добавление признаков по типам событий повысило результат до 0.1398.
Временные признаки повысили результат до 0.1944.
Признаки разнообразия поведения повысили результат до 0.3210.
Удаление полных дубликатов дополнительно повысило результат до 0.3373.
Добавление медианного интервала между событиями дало P@R0.7 = 0.4706
на локальной временной валидации.

Дополнительно проверялись признаки user-agent, платформы, интенсивности
событий, долей отдельных типов событий, стандартного отклонения интервалов
и распределения событий по половинам окна. Эти эксперименты не улучшили
результат и в финальную модель не вошли.

* user-agent: P@R0.7 = 0.3146;
* platform: 0.3228;
* интенсивность событий (events/hour): 0.3102;
* стандартное отклонение интервалов: 0.4195;
* доли отдельных типов событий: 0.3797;
* распределение событий по половинам окна: 0.4133;
* HistGradientBoosting вместо Random Forest: 0.4689.

Эти эксперименты не улучшили результат относительно лучшего значения 0.4706,
поэтому в финальную модель они не вошли.

## Финальная модель

CatBoostClassifier:

- iterations=500
- learning_rate=0.05
- depth=6
- loss_function=Logloss
- random_seed=0

После выбора признаков модель была обучена на всех доступных обучающих данных
и использована для получения score для test.

На скрытой проверке финальная версия получила P@R0.7 = 0.51108.

## Воспроизводимость

Python: 3.13.15

Зависимости указаны в requirements.txt.
Основные библиотеки: pandas, numpy, scikit-learn, scipy, catboost.
"""

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme)

print('README.md сохранён')